# Fracture Classification: YOLOv8 Backbone + PCA + KNN
Uses the pretrained YOLOv8s backbone as a frozen feature extractor,
PCA for dimensionality reduction, and KNN for binary classification
(fractured vs non-fractured).

## 1. Setup

In [ ]:
!pip install ultralytics scikit-learn tqdm matplotlib -q

In [ ]:
import torch
import numpy as np
from pathlib import Path
from PIL import Image
import torchvision.transforms as T
from ultralytics import YOLO
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

In [ ]:
# --- Config ---
IMAGES_DIR   = Path("datasets/images")        # contains Fractured/, Non_fractured/
DIST_DIR     = Path("Distribution")           # train.csv, valid.csv, test.csv
MODEL_PATH   = "yolov8s.pt"
IMGSZ        = 224
N_PCA        = 64
K_MAX        = 20
CV_FOLDS     = 5
SEED         = 42

## 2. Load Dataset Paths & Labels
Load fractured images using the Distribution CSVs (pre-defined splits).
Non-fractured images are split proportionally using the same ratios.

In [ ]:
def load_csv_ids(csv_path: Path) -> set:
    """Read image filenames from a Distribution CSV."""
    ids = set()
    with open(csv_path) as f:
        for line in f:
            line = line.strip()
            if not line or line == "image_id":
                continue
            ids.add(line)
    return ids


# --- Fractured splits (from CSVs) ---
frac_dir = IMAGES_DIR / "Fractured"

train_ids = load_csv_ids(DIST_DIR / "train.csv")
val_ids   = load_csv_ids(DIST_DIR / "valid.csv")
test_ids  = load_csv_ids(DIST_DIR / "test.csv")

frac_train = [frac_dir / n for n in train_ids if (frac_dir / n).exists()]
frac_val   = [frac_dir / n for n in val_ids   if (frac_dir / n).exists()]
frac_test  = [frac_dir / n for n in test_ids  if (frac_dir / n).exists()]

# --- Non-fractured split (proportional random split) ---
import random
random.seed(SEED)

nonfrac_dir  = IMAGES_DIR / "Non_fractured"
all_nonfrac  = sorted(nonfrac_dir.glob("*.jpg")) + sorted(nonfrac_dir.glob("*.png"))
random.shuffle(all_nonfrac)

n_total  = len(all_nonfrac)
n_val    = round(n_total * len(frac_val)   / (len(frac_train) + len(frac_val) + len(frac_test)))
n_test   = round(n_total * len(frac_test)  / (len(frac_train) + len(frac_val) + len(frac_test)))
n_train  = n_total - n_val - n_test

nf_train = all_nonfrac[:n_train]
nf_val   = all_nonfrac[n_train:n_train + n_val]
nf_test  = all_nonfrac[n_train + n_val:]

# --- Combine & assign labels (1 = fractured, 0 = non-fractured) ---
train_imgs = frac_train + nf_train
y_train    = np.array([1] * len(frac_train) + [0] * len(nf_train))

val_imgs   = frac_val + nf_val
y_val      = np.array([1] * len(frac_val) + [0] * len(nf_val))

test_imgs  = frac_test + nf_test
y_test     = np.array([1] * len(frac_test) + [0] * len(nf_test))

for name, imgs, y in [("train", train_imgs, y_train),
                      ("valid", val_imgs,   y_val),
                      ("test",  test_imgs,  y_test)]:
    print(f"{name:6s}: {len(imgs):5d} images | {y.sum()} fractured | {(y==0).sum()} non-fractured")

## 3. Feature Extraction via YOLOv8s Backbone
Hooks into **layer 9 (SPPF)** — the last backbone layer before the neck/head.
Global average pooling collapses spatial dims → 1D feature vector per image.

In [ ]:
yolo      = YOLO(MODEL_PATH)
nn_model  = yolo.model.to(DEVICE)
nn_model.eval()

# Hook on layer 9 (SPPF — last backbone layer)
_cache = {}
def _hook(module, inp, out):
    _cache["feat"] = out.detach()

hook_handle = nn_model.model[9].register_forward_hook(_hook)

transform = T.Compose([
    T.Resize((IMGSZ, IMGSZ)),
    T.ToTensor(),          # [0, 1]
])


def extract_features(image_paths):
    features = []
    for p in tqdm(image_paths, desc="Extracting"):
        img    = Image.open(p).convert("RGB")
        tensor = transform(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            nn_model(tensor)
        feat = _cache["feat"]          # [1, C, H, W]
        feat = feat.mean(dim=[2, 3])   # global average pool → [1, C]
        features.append(feat.squeeze().cpu().numpy())
    return np.array(features)


X_train_raw = extract_features(train_imgs)
X_val_raw   = extract_features(val_imgs)
X_test_raw  = extract_features(test_imgs)

hook_handle.remove()
print(f"\nRaw feature shape: {X_train_raw.shape}")

## 4. Standardise → PCA

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_raw)
X_val_sc   = scaler.transform(X_val_raw)
X_test_sc  = scaler.transform(X_test_raw)

pca = PCA(n_components=N_PCA, random_state=42)
X_train_pca = pca.fit_transform(X_train_sc)
X_val_pca   = pca.transform(X_val_sc)
X_test_pca  = pca.transform(X_test_sc)

cum_var = np.cumsum(pca.explained_variance_ratio_)
print(f"Variance explained by {N_PCA} components: {cum_var[-1]*100:.1f}%")

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, N_PCA + 1), cum_var, marker='o', markersize=3)
plt.axhline(0.95, color='r', linestyle='--', label='95% threshold')
plt.xlabel('Components')
plt.ylabel('Cumulative explained variance')
plt.title('PCA — Explained Variance')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Find Optimal k (Cross-Validation on Train Set)

In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)

k_values, f1_scores = [], []
for k in tqdm(range(1, K_MAX + 1), desc="Searching k"):
    knn    = KNeighborsClassifier(n_neighbors=k, metric='cosine', n_jobs=-1)
    scores = cross_val_score(knn, X_train_pca, y_train, cv=cv, scoring='f1')
    k_values.append(k)
    f1_scores.append(scores.mean())

best_k  = k_values[int(np.argmax(f1_scores))]
best_f1 = max(f1_scores)
print(f"Best k = {best_k}  |  CV F1 = {best_f1:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(k_values, f1_scores, marker='o')
plt.axvline(best_k, color='r', linestyle='--', label=f'Best k={best_k}')
plt.xlabel('k')
plt.ylabel('CV F1 (fractured class)')
plt.title('KNN — Finding Optimal k')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Train Final KNN & Evaluate

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k, metric='cosine', n_jobs=-1)
knn.fit(X_train_pca, y_train)

for split_name, X, y in [("Validation", X_val_pca, y_val), ("Test", X_test_pca, y_test)]:
    preds = knn.predict(X)
    print(f"\n{'='*40}")
    print(f"{split_name} Results")
    print('='*40)
    print(classification_report(y, preds, target_names=["Non-fractured", "Fractured"]))

    cm   = confusion_matrix(y, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Non-fractured", "Fractured"])
    disp.plot(cmap='Blues')
    plt.title(f'Confusion Matrix — {split_name}')
    plt.tight_layout()
    plt.show()

## 7. PCA Feature Space Visualisation (Test Set)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors  = ['steelblue', 'tomato']
labels  = ['Non-fractured', 'Fractured']

for cls in [0, 1]:
    mask = y_test == cls
    ax.scatter(X_test_pca[mask, 0], X_test_pca[mask, 1],
               c=colors[cls], label=labels[cls], alpha=0.6, s=20)

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('PCA Feature Space — Test Set')
ax.legend()
plt.tight_layout()
plt.show()